# Avant le TP2 : créer, alimenter et lire un topic

Le TP2 utilise trois objets de `confluent-kafka` : l'`AdminClient`, le producteur et le consommateur. Cette démonstration montre comment chacun répond, sur un topic jetable, `demo-tp2`, et donne à chaque fois la commande de terminal équivalente :

1. une opération d'administration renvoie un **futur**, et ses erreurs arrivent par une exception ;
2. le producteur met les messages en file : le **rapport de livraison** dit s'ils sont écrits ;
3. le consommateur peut lire une partition précise à partir d'un offset.

Chaque étape montre la version **terminal**, puis la version **Python**. Une étape qui ne fait que lire exécute les deux. Une étape qui modifie le cluster n'en exécute qu'une : l'autre est affichée à côté, **à ne pas lancer en plus**, sinon l'opération serait faite deux fois.

Sorties de référence : exécution du 12/11/2026 22:33 UTC sur le lab.

Les noms et les valeurs de la démonstration sont définis une seule fois, ici. Dans une ligne `!`, `$TOPIC` est remplacé par la variable Python du même nom ; dans un terminal de JupyterLab, définissez-la d'abord : `export TOPIC=demo-tp2`.

In [1]:
import os
import time

from confluent_kafka import Consumer, KafkaError, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic, OffsetSpec

BS = os.environ["BS"]
CONF = {"bootstrap.servers": BS}

TOPIC = "demo-tp2"
PARTITIONS = 2
RF = 3
GROUP = "demo-tp2"
MESSAGES = [("rouge", "premier"), ("vert", "deuxieme"), ("rouge", "troisieme")]

## 1. Administration : des futurs

`create_topics()` ne bloque pas : il renvoie un dictionnaire `{name: future}`. `result()` attend la réponse du cluster, et lève une `KafkaException` en cas d'échec. Le code d'erreur `TOPIC_ALREADY_EXISTS` n'est pas grave : il veut dire que la cellule a déjà été exécutée.

Équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic $TOPIC --partitions 2 --replication-factor 3
```

**En Python**

In [2]:
admin = AdminClient(CONF)
futures = admin.create_topics([NewTopic(TOPIC, num_partitions=PARTITIONS, replication_factor=RF)])
print(futures)

try:
    futures[TOPIC].result()
    print(f"{TOPIC} créé")
except KafkaException as e:
    if e.args[0].code() != KafkaError.TOPIC_ALREADY_EXISTS:
        raise
    print(f"{TOPIC} existe déjà")

{'demo-tp2': <Future at 0xffff8ef61450 state=running>}
demo-tp2 créé


Une erreur du cluster arrive avec un code, le même pour l'outil Java et pour Python. Créer une seconde fois le même topic ne modifie rien : on peut lancer les deux versions.

**En terminal** : l'outil Java affiche le message d'erreur.

In [3]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic $TOPIC --partitions 2 --replication-factor 3 2>&1 | head -1

Error while executing topic command : Topic 'demo-tp2' already exists.


**En Python** : le premier argument de l'exception porte le code et le message.

In [4]:
admin = AdminClient(CONF)
try:
    admin.create_topics([NewTopic(TOPIC, num_partitions=PARTITIONS, replication_factor=RF)])[TOPIC].result()
except KafkaException as e:
    error = e.args[0]
    print("Code :", error.name())
    print("Message :", error.str())

Code : TOPIC_ALREADY_EXISTS
Message : Topic 'demo-tp2' already exists.


## 2. Producteur : la file et le rapport de livraison

`produce()` range le message dans une file et rend la main tout de suite. `flush()` envoie la file et attend les réponses ; le rappel `on_delivery` est alors appelé une fois par message, avec l'erreur (ou `None`) et le message écrit.

Équivalent en terminal (**ne pas lancer en plus**) : une ligne `cle|valeur` par message.

```bash
printf 'rouge|premier\nvert|deuxieme\nrouge|troisieme\n' | docker exec -i kafka-tools kafka-console-producer.sh \
  --bootstrap-server $BS --topic $TOPIC --reader-property parse.key=true --reader-property 'key.separator=|'
```

**En Python** : exécutez cette cellule une seule fois, sinon les messages sont envoyés en double.

In [5]:
producer = Producer({**CONF, "log_level": 4})


def report(err, msg):
    if err is not None:
        print("Échec :", err)
    else:
        print("clé %s -> partition %d, offset %d" % (msg.key().decode(), msg.partition(), msg.offset()))


for key, value in MESSAGES:
    producer.produce(TOPIC, key=key, value=value, on_delivery=report)
print("Messages en file, rien n'est encore confirmé")

remaining = producer.flush(10)
print("Messages encore en file après flush() :", remaining)

Messages en file, rien n'est encore confirmé
clé rouge -> partition 1, offset 0
clé rouge -> partition 1, offset 1


clé vert -> partition 0, offset 0
Messages encore en file après flush() : 0


Les deux messages de clé `rouge` sont dans la même partition, l'un à la suite de l'autre ; celui de clé `vert` est dans l'autre. On le vérifie avec l'offset de fin de chaque partition, c'est-à-dire le nombre de messages écrits.

**En terminal**

In [6]:
!docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic $TOPIC

demo-tp2:0:1
demo-tp2:1:2


**En Python** : `list_offsets()` avec `OffsetSpec.latest()` demande l'offset de fin de chaque partition.

In [7]:
admin = AdminClient(CONF)
offset_requests = {TopicPartition(TOPIC, p): OffsetSpec.latest() for p in range(PARTITIONS)}
for tp, future in sorted(admin.list_offsets(offset_requests).items(), key=lambda x: x[0].partition):
    print("%s:%d:%d" % (tp.topic, tp.partition, future.result().offset))

demo-tp2:0:1
demo-tp2:1:2


## 3. Consommateur : lire une partition à partir d'un offset

On lit chaque partition depuis son début, sans groupe : rien n'est enregistré, la lecture peut être relancée.

**En terminal** : `--partition` et `--offset` désignent ce qu'on lit ; `--max-messages` et `--timeout-ms` arrêtent la lecture. `enable.auto.commit=false` empêche l'outil d'enregistrer sa position sous un groupe `console-consumer-NNNNN`, qui resterait ensuite dans la liste des groupes. Quand le délai expire, l'outil signale une `TimeoutException` sur sa sortie d'erreur : `2>/dev/null` la masque.

In [8]:
!docker exec kafka-tools kafka-console-consumer.sh --bootstrap-server $BS --topic $TOPIC --partition 0 --offset earliest --max-messages 10 --timeout-ms 5000 --command-property enable.auto.commit=false --formatter-property print.key=true --formatter-property print.partition=true --formatter-property print.offset=true 2>/dev/null
!docker exec kafka-tools kafka-console-consumer.sh --bootstrap-server $BS --topic $TOPIC --partition 1 --offset earliest --max-messages 10 --timeout-ms 5000 --command-property enable.auto.commit=false --formatter-property print.key=true --formatter-property print.partition=true --formatter-property print.offset=true 2>/dev/null

Partition:0	Offset:0	vert	deuxieme


Partition:1	Offset:0	rouge	premier
Partition:1	Offset:1	rouge	troisieme


**En Python** : `assign()` lit directement les partitions indiquées, à partir de l'offset donné, sans rééquilibrage de groupe. `consume(n, timeout)` renvoie une liste d'au plus `n` messages. Un élément de cette liste peut être une erreur : on la teste avec `m.error()` avant de lire le message. `close()` est dans un `finally` : le consommateur est fermé même si la lecture échoue.

In [9]:
consumer = Consumer({**CONF, "group.id": GROUP, "enable.auto.commit": False, "log_level": 4})
try:
    consumer.assign([TopicPartition(TOPIC, p, 0) for p in range(PARTITIONS)])
    for m in consumer.consume(10, timeout=10):
        if m.error():
            print("Erreur :", m.error())
            continue
        key = m.key().decode() if m.key() else None
        print(m.partition(), m.offset(), key, m.value().decode())
finally:
    consumer.close()

1 0 rouge premier
1 1 rouge troisieme
0 0 vert deuxieme


Dans une partition, les messages arrivent dans l'ordre de leurs offsets. Entre deux partitions, aucun ordre n'est garanti : c'est pourquoi la clé, qui fixe la partition, fixe aussi l'ordre.

## 4. Ménage

Le topic de démonstration est supprimé : le TP2 commence sur un cluster sans topic métier. La suppression est asynchrone : on interroge le cluster jusqu'à ce que le topic ait disparu.

Équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --delete --topic $TOPIC
```

**En Python**

In [10]:
admin = AdminClient(CONF)
try:
    admin.delete_topics([TOPIC])[TOPIC].result()
except KafkaException as e:
    if e.args[0].code() != KafkaError.UNKNOWN_TOPIC_OR_PART:
        raise
    print(f"{TOPIC} n'existe pas")

for _ in range(15):
    if TOPIC not in admin.list_topics(timeout=10).topics:
        break
    time.sleep(2)
print(sorted(admin.list_topics(timeout=10).topics))

['__consumer_offsets', 'loadgen']


## La méthode du TP2

Ouvrez `tps/jour1/tp2_topics/enonce.ipynb`. Ce TP crée des topics et envoie des messages : il modifie le cluster.

- Quand une commande modifie le cluster, son équivalent en ligne de commande est affiché **sans être exécuté**, avec la mention « ne pas lancer en plus du TODO ». C'est votre code Python qui fait l'opération.
- Exécutez une seule fois les cellules qui envoient des messages : les vérifications comptent les messages.
- Chaque vérification compare votre résultat à celui qu'aurait donné l'outil Java : clés, partitions, offsets.